In [82]:
class BasicTokenizer:
    def train(self, text, vocab_size, verbose=False):
        self.merges = dict()
        self.vocab = [i for i in range(256)]
        merge_num = vocab_size - 256
        byte_txt = list(text.encode("utf-8"))
        for i in range(merge_num):
            top_pair = count_pair(byte_txt)[0][0]
            self.vocab.append(top_pair)
            self.merges[top_pair] = i + 256
            byte_txt = self.merge(byte_txt, top_pair)
        
        self.rev_merges = {v: k for k, v in self.merges.items()}    

    def merge(self, text, pair):
        new_txt = []
        skip_next = False
        for i in range(len(text) - 1):
            if skip_next:
                skip_next = False
                continue
            if text[i:i+2] == list(pair):
                new_txt.append(self.merges[pair])
                skip_next = True
            else:
                new_txt.append(text[i])
        if not skip_next:
            new_txt.append(text[-1])
        return new_txt

    def encode(self, text):
        text = list(text.encode("utf-8"))
        for mer in self.merges:
            new_txt = []
            skip_next = False
            for pair in zip(text, text[1:]):
                if skip_next:
                    skip_next = False
                    continue
                if pair == mer:
                    new_txt.append(self.merges[pair])
                    skip_next = True
                else:
                    new_txt.append(pair[0])
            if not skip_next:
                new_txt.append(text[-1])
            text = new_txt.copy()

        return text

    def decode(self, ids):
        change = True
        while change:
            new_ids = []
            change = False
            for id in ids:
                if self.rev_merges.get(id):
                    new_ids += list(self.rev_merges[id])
                    change = True
                else:
                    new_ids.append(id)

            
            ids = new_ids.copy()
        text = bytes(ids).decode("utf-8")

        return text

In [83]:
tokenizer = BasicTokenizer()
tokenizer.train("hello uw ơ ????", 260)
tokenizer.encode("hello uw ơ ????")

[259, 111, 32, 117, 119, 32, 198, 161, 32, 256, 256]

In [84]:
def count_pair(text):
    """text in bytes"""
    count = dict()
    for a, b in zip(text, text[1:]):
        count[(a, b)] = count.get((a,b), 0) + 1

    return sorted(count.items(), key=lambda x: x[1], reverse=True)

In [85]:
byte_txt = "ă ơ ơ hsdf????".encode("utf-8")
byte_txt

list(merge(list(byte_txt), (63, 63)))

[196, 131, 32, 198, 161, 32, 198, 161, 32, 104, 115, 100, 102, '', '']

In [87]:
tokenizer.merges

{(63, 63): 256, (104, 101): 257, (257, 108): 258, (258, 108): 259}